In [ ]:
Laboratory work 2. ANN. Image Classification. Object Detection and Image Segmentation
Part 1. Regression using ANN
•
Use the regression problem introduced in Lab 1 (predicting real estate prices from historical transactions). If the data preprocessing from Part 1 has been performed properly, it may be reused here.
•
Implement a program and create your own ANN architecture (Multi-Layer Perceptron). The choice of number of layers, neurons, activation functions, and optimization method must be justified by comparison with respect to different combinations of hyperparameters.
•
Evaluate performance using metrics such as MAE (Mean Absolute Error), MAPE (Mean Absolute Percentage Error), RMSE (Root Mean Square Error), and 𝑅2score.
•
Visualize regression results (e.g., Predicted vs. Actual scatter plot with 45° line, residual distribution).
•
Provide insights on where ANN performs better or worse than KNN, Decision tree, Random forest (LD1), and discuss possible reasons
Part 2. Image Classification
•
Prepare dataset of given class objects (40-50 images per class). At least 10 images per class have to be unique (i.e. made by phone individually), others may be collected from publicly available sources.
•
Implement a program and create a model for solving an image classification problem. Use known architectures (their selection must be justified).
•
Train models and provide results (loss, accuracy, confusion matrix for train and test datasets) in the following cases:
    o
without transfer learning
o
with transfer learning (pre-trained model weights)
•
Visualize classification results (at least 5 examples of each class with information about actual class, predicted class, confidence level)
Part 3. Object Detection and Image Segmentation
•
Prepare dataset of given class objects (40–50 images per class). At least 10 images per class have to be unique (i.e., made by phone individually). It is allowed to reuse the dataset prepared for classification, however in this case the photos must be taken from a greater distance so that the segmented object does not exceed ~25% of the total image area. Additionally, several images must include multiple classes and/or multiple instances of the same class in one image.
•
Implement a program and create a model for solving an object detection and segmentation problem. Use known architectures (e.g., YOLO, Faster R-CNN, Mask R-CNN) — the selection must be justified.
•
Compare the results (mAP, IoU, precision, recall, segmentation accuracy) in the following cases:
o
without transfer learning
o
with transfer learning (pre-trained model weights)
•
Visualize detection and segmentation results on selected self-captured images (examples should cover all the classes and include images with multiple classes and/or multiple instances).

In [ ]:

# Laboratory Work 2 — ANN, Image Classification, Object Detection & Segmentation

** Course: ** Computational
Intelligence and Decision
Making(KTU)
** Student: ** _Rokas
Marcinkevičius_
** Topic
focus: ** Deep
Reinforcement
Learning
for Multi‑Asset Portfolio Optimization (thesis context) — * this lab focuses on ANN and CV tasks.*

> ** Instructions: ** This
notebook is a
structured
template
with clear ** PLACEHOLDER ** and ** TODO ** blocks you should fill.
> Keep
your
data
paths
local and reproducible.Prefer
deterministic
seeds
where
possible.

---

## 0. Environment & Reproducibility

- This
lab
can
be
completed
with ** TensorFlow / Keras ** or ** PyTorch **.The template shows ** Keras ** examples for simplicity and ** Ultralytics YOLO ** for detection / segmentation.
- If
you
prefer
PyTorch, swap
the
Keras
sections
accordingly.

> ** NOTE: ** Comment / uncomment
installs
depending
on
your
environment.

# OPTIONAL: installs (run only if needed)
# %pip install -q numpy pandas scikit-learn matplotlib
# %pip install -q tensorflow==2.*  # or: %pip install -q torch torchvision torchaudio
# %pip install -q ultralytics  # for YOLOv8 detection/segmentation

import os, sys, math, json, random, pathlib, itertools, time
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 5)  # adjust as needed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version)
print("Working dir:", os.getcwd())

---
# Part 1 — Regression using ANN (Real Estate Prices)

** Task: ** Reuse
the
regression
problem
from Lab

1(predict
real
estate
prices).
Evaluate ** MAE, MAPE, RMSE, R² ** , provide ** visualizations **, and ** compare ** with Lab 1 baselines (KNN, Decision Tree, Random Forest).
### 1.1 Data Loading & Preprocessing

> ** PLACEHOLDER: ** Provide
a
short
description
of
your
dataset, features, target, and preprocessing
steps
reused
from Lab

1.
> If
you
saved
a
preprocessed
CSV / Parquet in Lab
1, load
it
here
for reproducibility.

# TODO: set your local path(s)
DATA_PATH = "/mnt/data/real_estate_preprocessed.csv"  # PLACEHOLDER: change to your file
TARGET_COL = "price"  # PLACEHOLDER: adjust
ID_COLS = []  # columns to drop (IDs, leaks)

# Example loading (adjust to your schema)
df = pd.read_csv(DATA_PATH)

# Basic checks
print(df.shape)
df.head(3)
### 1.2 Train/Validation/Test Split

Use
a
stable
split(e.g., by
time or stratified
by
price
buckets if appropriate).

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Features/target
X = df.drop(columns=[TARGET_COL] + ID_COLS, errors="ignore")
y = df[TARGET_COL].astype(float)

# Simple split (you may prefer time-based split depending on data)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)

# Scale numeric features (adjust if you have categoricals/one-hot already)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_s[:1].shape, X_val_s[:1].shape, X_test_s[:1].shape
### 1.3 Model: Multi‑Layer Perceptron (Keras)

We’ll
define
a
small
factory
to
allow
hyperparameter
sweeps:
- Layers, units, activations
- Optimizer & learning
rate
- Dropout / batch‑norm(optional)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def build_mlp(input_dim: int,
              hidden_layers=(128, 64),
              activation="relu",
              output_activation=None,
              dropout=0.0,
              lr=1e-3):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
        if dropout and dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation=output_activation))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"]
    )
    return model


input_dim = X_train_s.shape[1]
model = build_mlp(input_dim, hidden_layers=(256, 128, 64), activation="relu", dropout=0.1, lr=1e-3)
model.summary()
### 1.4 Training & Early Stopping
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)

hist = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=300,
    batch_size=64,
    callbacks=[early],
    verbose=1
)

# Plot training curves
plt.figure()
plt.plot(hist.history["loss"], label="train_loss")
plt.plot(hist.history["val_loss"], label="val_loss")
plt.xlabel("epoch");
plt.ylabel("MSE");
plt.legend();
plt.title("Training Curves")
plt.show()
### 1.5 Metrics: MAE, MAPE, RMSE, R²
from sklearn.metrics import mean_absolute_error, r2_score


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    eps = 1e-8
    return np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100.0


def rmse(y_true, y_pred):
    return math.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))


y_pred = model.predict(X_test_s).ravel()
metrics = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "MAPE_%": mape(y_test, y_pred),
    "RMSE": rmse(y_test, y_pred),
    "R2": r2_score(y_test, y_pred)
}
metrics
### 1.6 Visualizations (Required)

- ** Predicted
vs
Actual **
with 45° line
- ** Residual
distribution **
# Predicted vs Actual
plt.figure()
plt.scatter(y_test, y_pred, s=12)
m = min(y_test.min(), y_pred.min())
M = max(y_test.max(), y_pred.max())
plt.plot([m, M], [m, M])
plt.xlabel("Actual");
plt.ylabel("Predicted");
plt.title("Predicted vs Actual")
plt.show()

# Residuals
res = y_test - y_pred
plt.figure()
plt.hist(res, bins=30)
plt.xlabel("Residual");
plt.ylabel("Count");
plt.title("Residual Distribution")
plt.show()
### 1.7 Hyperparameter Comparison (Justification)

> ** TODO: ** Populate
`sweep_configs` and run
comparisons(few
sensible
combos).
> Provide
a
short
written
justification
based
on
results.

from collections import OrderedDict

sweep_configs = [
    {"hidden_layers": (128, 64), "lr": 1e-3, "dropout": 0.0},
    {"hidden_layers": (256, 128, 64), "lr": 1e-3, "dropout": 0.1},
    {"hidden_layers": (512, 256, 128), "lr": 5e-4, "dropout": 0.1},
]

results = []
for cfg in sweep_configs:
    m = build_mlp(input_dim, hidden_layers=cfg["hidden_layers"], lr=cfg["lr"], dropout=cfg["dropout"])
    h = m.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=200, batch_size=64,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
              verbose=0)
    yp = m.predict(X_test_s).ravel()
    row = OrderedDict(config=str(cfg))
    row["MAE"] = mean_absolute_error(y_test, yp)
    row["MAPE_%"] = mape(y_test, yp)
    row["RMSE"] = rmse(y_test, yp)
    row["R2"] = r2_score(y_test, yp)
    results.append(row)

hp_df = pd.DataFrame(results)
hp_df.sort_values("RMSE", inplace=True)
hp_df
### 1.8 Comparison with Lab 1 Baselines (KNN, DT, RF)

> ** PLACEHOLDER: ** Either
re - run
baselines
here
quickly or ** import their

metrics **
from Lab

1(CSV / JSON).Provide
a
short
discussion
of
where
ANN
does
better / worse and why.

# OPTION A: If you saved baseline metrics to CSV in Lab 1
# baseline_df = pd.read_csv("/mnt/data/lab1_baselines.csv")  # columns: model, MAE, MAPE_%, RMSE, R2
# display(baseline_df)

# OPTION B: (Quick) Recompute simple baselines if data is small
# from sklearn.neighbors import KNeighborsRegressor
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.ensemble import RandomForestRegressor
# ...

# PLACEHOLDER: final comparison table
# combined = pd.concat([hp_df.assign(model="ANN (best)").head(1), baseline_df], ignore_index=True)
# combined
---
# Part 2 — Image Classification

** Task: ** Prepare
a
dataset(40–50
images
per


class ; ≥10 unique photos per class ).
Train models ** without ** and ** with ** transfer learning; report ** loss / accuracy **, ** confusion matrices ** (train / test), and ** visualize predictions ** (≥5 examples per class ).

### 2.1 Dataset Structure & Loader

Expected folder layout (example):
    ```
    data / classification /
    train /
    classA / img001.jpg...
    classB / ...


val /
classA / ...
classB / ...
test /
classA / ...
classB / ...
```
  > ** PLACEHOLDER: ** update
paths, classes, and image
size.

from tensorflow.keras.preprocessing import image_dataset_from_directory

CLS_DATA_DIR = "/mnt/data/classification"  # PLACEHOLDER
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = image_dataset_from_directory(
os.path.join(CLS_DATA_DIR, "train"),
validation_split = None,
image_size = IMG_SIZE,
batch_size = BATCH_SIZE,
shuffle = True,
seed = SEED
)
val_ds = image_dataset_from_directory(
os.path.join(CLS_DATA_DIR, "val"),
image_size = IMG_SIZE,
batch_size = BATCH_SIZE,
shuffle = False
)
test_ds = image_dataset_from_directory(
os.path.join(CLS_DATA_DIR, "test"),
image_size = IMG_SIZE,
batch_size = BATCH_SIZE,
shuffle = False
)

class_names = train_ds.class_names
num_classes = len(class_names)
class_names, num_classes
### 2.2 Model **without** Transfer Learning (from scratch)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])


def build_cnn_scratch(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape + (3,))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1. / 255)(x)
    # Simple CNN backbone
    x = layers.Conv2D(32, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


cnn_scratch = build_cnn_scratch(IMG_SIZE, num_classes)
cnn_scratch.summary()

history_scratch = cnn_scratch.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    verbose=1
)
### 2.3 Model **with** Transfer Learning (pretrained)
base = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False  # first stage: freeze

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model_tl = keras.Model(inputs, outputs)
model_tl.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])

history_tl = model_tl.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1)

# Optional fine-tuning stage
base.trainable = True
for layer in base.layers[:-30]:  # unfreeze last ~30 layers
    layer.trainable = False

model_tl.compile(optimizer=keras.optimizers.Adam(1e-4),
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])

history_ft = model_tl.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1)

### 2.4 Confusion Matrices (train & test) and Sample Visualizations
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


def ds_to_numpy(ds):
    y_true, y_prob = [], []
    for xb, yb in ds:
        y_true.append(yb.numpy())
        y_prob.append(model_tl.predict(xb, verbose=0))
    y_true = np.concatenate(y_true, axis=0)
    y_prob = np.concatenate(y_prob, axis=0)
    y_pred = np.argmax(y_prob, axis=1)
    return y_true, y_pred, y_prob


# Confusion matrix on test
y_true, y_pred, y_prob = ds_to_numpy(test_ds)
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots()
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Test Confusion Matrix")
plt.show()


# Visualize 5 examples per class (actual vs predicted with confidence)
def show_examples_per_class(ds, per_class=5):
    picks = {i: 0 for i in range(num_classes)}
    for xb, yb in ds:
        probs = model_tl.predict(xb, verbose=0)
        preds = np.argmax(probs, axis=1)
        confs = np.max(probs, axis=1)
        for i in range(xb.shape[0]):
            cls = int(yb[i].numpy())
            if picks[cls] < per_class:
                img = xb[i].numpy().astype("uint8")
                true_name = class_names[cls]
                pred_name = class_names[preds[i]]
                conf = confs[i]
                plt.figure()
                plt.imshow(img)
                plt.axis("off")
                plt.title(f"Actual: {true_name} | Pred: {pred_name} ({conf:.2f})")
                plt.show()
                picks[cls] += 1
        if all(v >= per_class for v in picks.values()):
            break


show_examples_per_class(test_ds, per_class=5)
---
# Part 3 — Object Detection and Image Segmentation

** Task: ** Prepare
a
dataset(40–50
images
per


class ; ≥10 unique).You may reuse classification images, but from a ** greater distance ** so the object occupies ≤25 %of the image.Include some images with ** multiple classes / instances **.

Train a model ** without ** and ** with ** transfer learning (pretrained weights).Compare ** mAP, IoU, precision, recall, segmentation accuracy **.Visualize results.

### 3.1 Dataset Prep (YOLO format recommended)

Expected layout ( for detection):
    ```
    data / detect /
    images / train / *.jpg


images / val / *.jpg
images / test / *.jpg
labels / train / *.txt  # YOLO txt with class_id x_center y_center w h (normalized)
labels / val / *.txt
labels / test / *.txt
```
For ** segmentation **, YOLOv8
uses
polygon
masks in the
same
txt
files(
with additional coordinates).

> ** PLACEHOLDER:**
Provide
the
path
to
your
`data.yaml`
describing
classes and paths.

# Example YOLO data.yaml (EDIT and save for your project)
yolo_data_yaml = {
"path": "/mnt/data/detect",  # root dataset dir (PLACEHOLDER)
"train": "images/train",
"val": "images/val",
"test": "images/test",
"names": ["classA", "classB"]  # PLACEHOLDER: your classes
}
yaml_path = "/mnt/data/yolo_data.yaml"
with open(yaml_path, "w") as f:
    import yaml
yaml.safe_dump(yolo_data_yaml, f)
print("Wrote:", yaml_path)
### 3.2 Training — **Detection** (YOLOv8n)
Train
from scratch ( ** no
transfer
learning **) and with pretrained weights.

from ultralytics import YOLO

# No transfer learning (random init): pass weights=None or a blank new model
detect_model_scratch = YOLO("yolov8n.pt")  # Start from small model; to simulate "no TL", set pretrained=False in train
detect_model_scratch.train(
data = "/mnt/data/yolo_data.yaml",  # EDIT to your yaml_path
epochs = 50,
imgsz = 640,
batch = 16,
pretrained = False,  # <- key difference
name = "detect_scratch"
)

# With transfer learning (pretrained on COCO)
detect_model_tl = YOLO("yolov8n.pt")
detect_model_tl.train(
data = "/mnt/data/yolo_data.yaml",
epochs = 50,
imgsz = 640,
batch = 16,
pretrained = True,
name = "detect_tl"
)
### 3.3 Training — **Segmentation** (YOLOv8n‑seg)
# Segmentation models: yolov8n-seg, yolov8s-seg, ...
seg_model_scratch = YOLO("yolov8n-seg.pt")
seg_model_scratch.train(
data = "/mnt/data/yolo_data.yaml",
epochs = 50,
imgsz = 640,
batch = 8,
pretrained = False,
name = "seg_scratch"
)

seg_model_tl = YOLO("yolov8n-seg.pt")
seg_model_tl.train(
data = "/mnt/data/yolo_data.yaml",
epochs = 50,
imgsz = 640,
batch = 8,
pretrained = True,
name = "seg_tl"
)
### 3.4 Evaluation — mAP, IoU, Precision, Recall, Segmentation Accuracy
# Evaluate on test split (Ultralytics provides metrics including mAP)
detect_metrics_scratch = detect_model_scratch.val(data="/mnt/data/yolo_data.yaml", split="test", plots=True,
                                                  save_json=True)
detect_metrics_tl = detect_model_tl.val(data="/mnt/data/yolo_data.yaml", split="test", plots=True, save_json=True)

seg_metrics_scratch = seg_model_scratch.val(data="/mnt/data/yolo_data.yaml", split="test", plots=True, save_json=True)
seg_metrics_tl = seg_model_tl.val(data="/mnt/data/yolo_data.yaml", split="test", plots=True, save_json=True)

print("Detect (scratch):", detect_metrics_scratch.results_dict if hasattr(detect_metrics_scratch,
                                                                          "results_dict") else detect_metrics_scratch)
print("Detect (TL):",
      detect_metrics_tl.results_dict if hasattr(detect_metrics_tl, "results_dict") else detect_metrics_tl)
print("Seg (scratch):",
      seg_metrics_scratch.results_dict if hasattr(seg_metrics_scratch, "results_dict") else seg_metrics_scratch)
print("Seg (TL):", seg_metrics_tl.results_dict if hasattr(seg_metrics_tl, "results_dict") else seg_metrics_tl)
### 3.5 Visualization — Predictions on Self‑Captured Images

> ** TODO: ** Place
a
few
of
your
own
photos in a
folder and run
inference
below
for both detection and segmentation.Include examples with multiple classes / instances.

TEST_IMG_DIR = "/mnt/data/my_photos"  # PLACEHOLDER: folder with a few sample images

os.makedirs("/mnt/data/preds_detect", exist_ok=True)
detect_model_tl.predict(
source = TEST_IMG_DIR,
imgsz = 640,
save = True,
project = "/mnt/data",
name = "preds_detect"
)

os.makedirs("/mnt/data/preds_seg", exist_ok=True)
seg_model_tl.predict(
source = TEST_IMG_DIR,
imgsz = 640,
save = True,
project = "/mnt/data",
name = "preds_seg"
)

print("Saved predictions to /mnt/data/preds_detect and /mnt/data/preds_seg")
- --
## 4. Discussion & Conclusions

- ** Regression(ANN
vs
KNN / DT / RF): **
- ** Where
ANN
wins: ** _PLACEHOLDER_(e.g., nonlinear
relationships, richer
feature
interactions).
- ** Where
ANN
struggles: ** _PLACEHOLDER_(e.g., small
data, overfitting
without
regularization).
- ** Why: ** _PLACEHOLDER_.

             - ** Classification(without
vs
with TL):**
- Summarize
accuracy and confusion
matrices.Where
does
transfer
learning
help
most? Any class imbalance?

- ** Detection / Segmentation(scratch
vs
TL): **
- Compare
mAP / IoU / precision / recall.Discuss
data
size
sufficiency and augmentation.

- ** Reproducibility: **
- Note
seeds, versions, and any
randomness
sources.Attach
`data.yaml`, environment
info if required.
